# 03. Çok Başlı Dikkat Haritaları ve Maskeleme Görselleştirmesi

Bu notebook, "Attention Is All You Need" (Vaswani et al., 2017) modelindeki çoklu dikkat başlarının
(Multi-Head Attention) ağırlık dağılımlarını ve causal / padding maskelemenin etkisini görselleştirir.

In [ ]:
import sys
sys.path.append("..")
import torch
import matplotlib.pyplot as plt
from src import MultiHeadAttention, generate_square_subsequent_mask

# Hiperparametreler
d_model = 64
num_heads = 4
seq_len = 6

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=0.0)
mha.eval()

# Örnek dizi girdisi: [Batch=1, Seq_Len=6, d_model=64]
torch.manual_seed(101)
x = torch.randn(1, seq_len, d_model)
tokens = ["The", "animal", "didn't", "cross", "the", "street"]
print("Token Listesi:", tokens)

## 1. Serbest Öz-Dikkat (Unmasked Self-Attention)
Her token, cümlenin hem solundaki hem de sağındaki tüm token'lara serbestçe dikkat yöneltebilir (Encoder stili).

In [ ]:
with torch.no_grad():
    _, attn_weights = mha(x, x, x)
# attn_weights: [1, num_heads, seq_len, seq_len]
attn_unmasked = attn_weights.squeeze(0).numpy()

fig, axes = plt.subplots(1, num_heads, figsize=(16, 4))
for h in range(num_heads):
    ax = axes[h]
    im = ax.imshow(attn_unmasked[h], cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Baş (Head) {h+1}')
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(tokens, rotation=45)
    ax.set_yticklabels(tokens)

plt.suptitle('Encoder Öz-Dikkat Ağırlıkları (Tüm Yönler Açık)', fontsize=14, y=1.05)
plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='Dikkat Olasılığı')
plt.show()

## 2. Nedensel Maskeli Öz-Dikkat (Causal / Look-Ahead Masked Self-Attention)
Decoder üretiminde, gelecek token'lar $-\infty$ ile maskelenir. Dikkat matrisinin üst üçgeni sıfırlanır.

In [ ]:
causal_mask = generate_square_subsequent_mask(seq_len)
print("Causal Maske Tensörü:\n", causal_mask.squeeze(0).int())

with torch.no_grad():
    _, attn_masked = mha(x, x, x, mask=causal_mask.unsqueeze(1))
attn_masked = attn_masked.squeeze(0).numpy()

fig, axes = plt.subplots(1, num_heads, figsize=(16, 4))
for h in range(num_heads):
    ax = axes[h]
    im = ax.imshow(attn_masked[h], cmap='Purples', vmin=0, vmax=1)
    ax.set_title(f'Maskeli Baş {h+1}')
    ax.set_xticks(range(seq_len))
    ax.set_yticks(range(seq_len))
    ax.set_xticklabels(tokens, rotation=45)
    ax.set_yticklabels(tokens)

plt.suptitle('Decoder Causal Maskeli Öz-Dikkat Ağırlıkları (Üst Üçgen Engelli)', fontsize=14, y=1.05)
plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='Dikkat Olasılığı')
plt.show()